# Lesson 02: Classification with PyTorch

This notebook covers:
- Binary classification (2 classes)
- Multi-class classification (4 classes)
- Non-linear activation functions (ReLU)
- Different loss functions (BCEWithLogitsLoss, CrossEntropyLoss)
- Decision boundary visualization

## Setup and Imports

In [ ]:
# Load the required libraries
import torch
from torch import nn  # Neural network building blocks
import numpy as np
import matplotlib.pyplot as plt
import time  # For timestamping random seeds

# Display PyTorch version for debugging compatibility
torch.__version__

In [ ]:
# Setup device-agnostic code
# This allows code to run on GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{device} is in use")

## Part 1: Creating a Binary Classification Dataset

We'll create a "circles" dataset where the task is to classify points as belonging to the inner or outer circle.

In [ ]:
# Import scikit-learn for dataset creation
import sklearn

In [ ]:
# Import the make_circles function to create circular dataset
from sklearn.datasets import make_circles

In [ ]:
# Create a binary classification dataset with 1000 samples
# This creates two concentric circles - a classic non-linear problem
n_samples = 1000

# make_circles creates 2D points in two classes (inner and outer circle)
X, y = make_circles(
    n_samples,
    noise=0.03,  # Add slight randomness to points
    random_state=42  # For reproducibility
)

In [ ]:
# Check the number of samples in features (X) and labels (y)
# Both should be 1000
len(X), len(y)

In [ ]:
# Display first 5 samples
# X: 2D coordinates (x1, x2) for each point
# y: class labels (0 for inner circle, 1 for outer circle)
print(f"Print first five samples of X:\n {X[:5]}\n")
print(f"Print first five samples of y:\n {y[:5]}")

In [ ]:
# Import pandas for data visualization
import pandas as pd

In [ ]:
# Create a DataFrame to better visualize the data
# X1 and X2 are the two feature dimensions
# label is the class (0 or 1)
circles = pd.DataFrame({
    "X1": X[:, 0],
    "X2": X[:, 1],
    "label": y
})
circles.head(10)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Visualize the circular dataset
# Different colors (red/blue) represent different classes
# Note: A straight line cannot separate these circles (non-linear problem)
plt.figure(figsize=(4, 4))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="RdYlBu")
plt.show()

## Data Preparation

In [ ]:
# Check shapes before converting to tensors
X.shape, y.shape

In [ ]:
# Convert NumPy arrays to PyTorch tensors
# PyTorch models require tensor inputs, not NumPy arrays
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

In [ ]:
# Verify the data types after conversion
# Should be torch.Tensor with dtype float32
type(X), X.dtype, y.dtype

## Train-Test Split

In [ ]:
# Import train_test_split for splitting data
from sklearn.model_selection import train_test_split

In [ ]:
# Split data into training (80%) and testing (20%) sets
# Using current time as random_state for variability
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=int(time.time())
)

In [ ]:
# Check training set shapes
# Should be approximately 800 samples
X_train.shape, y_train.shape

In [ ]:
# Check test set shapes
# Should be approximately 200 samples
X_test.shape, y_test.shape

In [ ]:
# Confirm device for computations
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Display the device being used
device

In [ ]:
# Verify input feature dimension
# (800, 2) means 800 samples with 2 features each
X_train.shape

## Part 2: Building a Simple Neural Network (Without Non-linearity)

In [ ]:
# Define a simple neural network with 2 linear layers
# Note: This model has NO activation functions (no non-linearity)
class CircleModelV0(nn.Module):

    def __init__(self):
        super().__init__()

        # Create two linear (fully connected) layers
        # Layer 1: Takes 2 input features and expands to 5 hidden features
        self.layer1 = nn.Linear(in_features=2, out_features=5)
        # Layer 2: Takes 5 hidden features and outputs 1 value (binary classification)
        self.layer2 = nn.Linear(in_features=5, out_features=1)

    def forward(self, x):
        """
        Define the forward pass through the network.
        Data flows: x -> layer1 -> layer2 -> output
        """
        return self.layer2(self.layer1(x))  # Composition of linear transformations
    
# Create an instance of the model and move it to the target device (CPU or GPU)
model_0 = CircleModelV0().to(device)

In [ ]:
# Verify that the model parameters are on the correct device
next(model_0.parameters()).device

In [ ]:
# Alternative way to create the same model using nn.Sequential
# nn.Sequential chains layers together in order
# This is more compact than subclassing nn.Module
model_0 = nn.Sequential(
    nn.Linear(in_features=2, out_features=5),
    nn.Linear(in_features=5, out_features=1)
).to(device)

model_0

In [ ]:
# Display the model's learnable parameters (weights and biases)
model_0.state_dict()

In [ ]:
# Make predictions with the untrained model
# These will be random since the model hasn't learned anything yet
with torch.inference_mode():
    # Move training data to device before passing to model
    untrained_preds = model_0(X_train.to(device))

# Display prediction statistics
print(f"Length of predictions: {len(untrained_preds)}, Shape: {untrained_preds.shape}")
print(f"Length of test samples: {len(X_test)}, Shape: {X_test.shape}")
print(f"\nFirst 10 predictions:\n{untrained_preds[:10]}")
print(f"\nFirst 10 labels:\n{y_test[:10]}")

## Loss Function and Optimizer for Binary Classification

In [ ]:
# Setup loss function for binary classification
# BCEWithLogitsLoss = Binary Cross Entropy with Logits
# "WithLogits" means it expects raw outputs (not probabilities)
# It combines sigmoid activation + BCE loss in one operation (more numerically stable)
loss_fn = nn.BCEWithLogitsLoss()

# Setup optimizer - Stochastic Gradient Descent
optimizer = torch.optim.SGD(
    params=model_0.parameters(),
    lr=0.1  # Learning rate
)

## Accuracy Function

In [ ]:
# Define a function to calculate classification accuracy
def accuracy_fn(y_true, y_pred):
    """
    Calculate percentage of correct predictions.
    
    Args:
        y_true: Ground truth labels
        y_pred: Predicted labels
        
    Returns:
        Accuracy as a percentage (0-100)
    """
    # Count how many predictions match the true labels
    correct = torch.eq(y_true, y_pred).sum().item()
    # Calculate percentage
    acc = 100 * correct / len(y_pred)
    return acc

## Understanding Logits, Probabilities, and Predictions

In [ ]:
# Demonstrate the transformation: logits -> probabilities -> predictions
model_0.eval()

with torch.inference_mode():
    # Get raw outputs (logits) from the model
    # Logits are unbounded values (can be any number)
    y_logits = model_0(X_test.to(device))[:5]

print(y_logits)

In [ ]:
# Display the true test labels for comparison
y_test[:5]

In [ ]:
# Convert logits to probabilities using sigmoid function
# Sigmoid squashes values to range (0, 1)
# Values closer to 1 = higher confidence for class 1
# Values closer to 0 = higher confidence for class 0
y_preds_probs = torch.sigmoid(y_logits)
print(y_preds_probs)

In [ ]:
# Convert probabilities to binary predictions (0 or 1)
# torch.round() rounds to nearest integer:
# - Probabilities >= 0.5 become 1 (outer circle)
# - Probabilities < 0.5 become 0 (inner circle)
y_preds = torch.round(y_preds_probs)

# Alternative: Calculate prediction labels directly from test set
y_pred_labels = torch.round(torch.sigmoid(model_0(X_test.to(device))[:5]))

# Verify both methods produce the same results
print(torch.eq(y_preds.squeeze(), y_pred_labels.squeeze()))

y_preds.squeeze()

## Training Loop for Model V0 (Linear Model - Will Fail on Non-linear Data)

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(int(time.time()))
torch.cuda.manual_seed(int(time.time()))

# Set number of training epochs
epochs = 100

# Move all data to the target device (CPU or GPU)
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

# Training loop
for epoch in range(epochs):

    ### TRAINING PHASE ###
    model_0.train()  # Set model to training mode

    # 1. Forward pass - get model predictions
    y_logits = model_0(X_train).squeeze()  # Remove extra dimension
    y_pred = torch.round(torch.sigmoid(y_logits))  # Convert to binary predictions

    # 2. Calculate loss and accuracy
    loss = loss_fn(y_logits, y_train)  # BCEWithLogitsLoss expects logits, not probabilities
    acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

    # 3. Zero gradients from previous iteration
    optimizer.zero_grad()

    # 4. Backpropagation - compute gradients
    loss.backward()

    # 5. Update parameters using gradients
    optimizer.step()

    ### TESTING PHASE ###
    model_0.eval()  # Set model to evaluation mode
    with torch.inference_mode():
        # 1. Forward pass on test data
        test_logits = model_0(X_test).squeeze()
        test_pred = torch.round(torch.sigmoid(test_logits))

        # 2. Calculate test loss and accuracy
        test_loss = loss_fn(test_logits, y_test)
        test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

        # Print progress every 10 epochs
        if epoch > 1 and epoch % 10 == 0:
            print(f"Epoch: {epoch} | Loss: {loss:.5f}, Acc: {acc:.2f}% | Test Loss: {test_loss:.5f}, Test Acc: {test_acc:.2f}%")

## Visualizing Decision Boundaries

In [ ]:
# Download helper functions for visualization
import requests
from pathlib import Path

# Check if helper file already exists
if Path("helper_functions.py").is_file():
    print("helper_functions.py already exists, skipping downloading")
else:
    print("Downloading helper_functions.py")
    request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
    with open("helper_functions.py", "wb") as f:
        f.write(request.content)

# Import visualization functions
from helper_functions import plot_predictions, plot_decision_boundary

In [ ]:
# Plot decision boundaries for training and test sets
# The decision boundary shows where the model draws the line between classes
# For the circles dataset, we need a curved boundary (non-linear)
# Model V0 will show a straight line (linear) - inadequate for this problem
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title("Train")
plot_decision_boundary(model_0, X_train, y_train)
plt.subplot(1, 2, 2)
plt.title("Test")
plot_decision_boundary(model_0, X_test, y_test)

## Part 3: Improving the Model - Adding More Layers (Still Linear)

In [ ]:
# Create a deeper model with an extra hidden layer
# Note: Still no activation functions, so still effectively linear!
class CircleModelV1(nn.Module):
    def __init__(self):
        super().__init__()
        # Expanded architecture: 2 -> 10 -> 10 -> 1
        self.layer_1 = nn.Linear(in_features=2, out_features=10)
        self.layer_2 = nn.Linear(in_features=10, out_features=10)  # Extra layer
        self.layer_3 = nn.Linear(in_features=10, out_features=1)
        
    def forward(self, x):
        """
        Forward pass through three linear layers.
        Note: Composition of linear transformations is still linear!
        This won't help with non-linear problems.
        """
        # These two approaches are equivalent:
        # Method 1 (explicit):
        # z = self.layer_1(x)
        # z = self.layer_2(z)
        # z = self.layer_3(z)
        # return z
        
        # Method 2 (compact):
        return self.layer_3(self.layer_2(self.layer_1(x)))

# Create instance and move to device
model_1 = CircleModelV1().to(device)
model_1

In [ ]:
# Setup loss and optimizer for model_1
# Using same loss function and optimizer as before
loss_fn = nn.BCEWithLogitsLoss()  # Binary Cross Entropy with Logits
optimizer = torch.optim.SGD(model_1.parameters(), lr=0.1)

In [ ]:
# Train model_1 for more epochs
torch.manual_seed(42)

epochs = 1000  # Train for longer to see if it improves

# Ensure data is on target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

for epoch in range(epochs + 1):
    ### TRAINING ###
    model_1.train()
    
    # 1. Forward pass
    y_logits = model_1(X_train).squeeze()
    # Convert logits -> probabilities -> prediction labels
    y_pred = torch.round(torch.sigmoid(y_logits))

    # 2. Calculate loss and accuracy
    loss = loss_fn(y_logits, y_train)
    acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

    # 3. Optimizer zero grad
    optimizer.zero_grad()

    # 4. Loss backwards
    loss.backward()

    # 5. Optimizer step
    optimizer.step()

    ### TESTING ###
    model_1.eval()
    with torch.inference_mode():
        # 1. Forward pass
        test_logits = model_1(X_test).squeeze()
        test_pred = torch.round(torch.sigmoid(test_logits))
        
        # 2. Calculate loss and accuracy
        test_loss = loss_fn(test_logits, y_test)
        test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

    # Print progress every 100 epochs
    if epoch % 100 == 0:
        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

In [ ]:
# Visualize decision boundaries for model_1
# Still shows linear boundary despite more layers!
# Multiple linear layers without activation = still just linear transformation
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title("Train")
plot_decision_boundary(model_1, X_train, y_train)
plt.subplot(1, 2, 2)
plt.title("Test")
plot_decision_boundary(model_1, X_test, y_test)

## Testing on Linear Regression Dataset

Let's verify that our neural network can still handle simple linear problems.

In [ ]:
# Create a simple linear regression dataset
# This will help us verify that our model architecture works for linear problems
weight = 0.7
bias = 0.3
start = 0
end = 1
step = 0.01

# Create linear data with some noise
X_regression = torch.arange(start, end, step).unsqueeze(dim=1)
# y = wx + b + noise
y_regression = weight * X_regression + bias + 0.05 * torch.rand(np.shape(X_regression)[0]).unsqueeze(dim=1)

# Split into train and test sets
data_split = int(0.8 * np.shape(X_regression)[0])

X_train_reg = X_regression[:data_split]
y_train_reg = y_regression[:data_split]

X_test_reg = X_regression[data_split:]
y_test_reg = y_regression[data_split:]

In [ ]:
# Visualize the linear dataset
plt.scatter(X_train_reg, y_train_reg, label="training data")
plt.scatter(X_test_reg, y_test_reg, label="test data")
plt.legend()
plt.show()

In [ ]:
# Plot predictions (currently none)
plot_predictions(
    train_data=X_train_reg,
    train_labels=y_train_reg,
    test_data=X_test_reg,
    test_labels=y_test_reg,
)

In [ ]:
# Create a neural network for regression
# Architecture: 1 input -> 10 hidden -> 10 hidden -> 1 output
model_2 = nn.Sequential(
    nn.Linear(in_features=1, out_features=10),
    nn.Linear(in_features=10, out_features=10),
    nn.Linear(in_features=10, out_features=1),
).to(device)

model_2

In [ ]:
# Setup loss function and optimizer for regression
# Using L1Loss (Mean Absolute Error) instead of BCE
loss_fn = nn.L1Loss()
optimizer = torch.optim.SGD(
    params=model_2.parameters(),
    lr=0.01
)

In [ ]:
# Train the regression model
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Move data to device
X_train_reg, y_train_reg = X_train_reg.to(device), y_train_reg.to(device)
X_test_reg, y_test_reg = X_test_reg.to(device), y_test_reg.to(device)

# Track loss over epochs
epochs_dis = []
test_loss_dis = []

epochs = 150

for epoch in range(epochs):
    # Training
    model_2.train()
    y_pred = model_2(X_train_reg)
    loss = loss_fn(y_pred, y_train_reg)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Testing
    model_2.eval()
    with torch.inference_mode():
        test_pred = model_2(X_test_reg)
        test_loss = loss_fn(test_pred, y_test_reg)

    # Store for plotting
    epochs_dis.append(epoch)
    test_loss_dis.append(test_loss.to('cpu'))  # Move to CPU for plotting
    
    if epoch % 10 == 0:
        print(f"Epoch: {epoch} | Loss: {loss:.5f} | Test loss: {test_loss:.5f}")

In [ ]:
# Plot test loss over training
# Every 10th epoch to reduce clutter
n = 10

plt.plot(epochs_dis[::n], test_loss_dis[::n])
plt.show()

In [ ]:
# Make predictions and visualize results
model_2.eval()

with torch.inference_mode():
    y_preds = model_2(X_test_reg)

# Plot predictions alongside actual data
# Need to move tensors to CPU for matplotlib
plot_predictions(
    train_data=X_train_reg.cpu(),
    train_labels=y_train_reg.cpu(),
    test_data=X_test_reg.cpu(),
    test_labels=y_test_reg.cpu(),
    predictions=y_preds.cpu()
)

## Part 4: Adding Non-linearity with ReLU Activation

The key to solving non-linear problems: activation functions!

In [ ]:
# Recreate circles dataset
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles

n_samples = 1000

X, y = make_circles(
    n_samples=1000,
    noise=0.03,
    random_state=42,
)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdBu);

In [ ]:
# Convert to tensors and split into train and test sets
import torch
from sklearn.model_selection import train_test_split

# Turn data into tensors
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train[:5], y_train[:5]

In [ ]:
# Create a model with NON-LINEAR activation functions
# This is the key difference from previous models!
from torch import nn

class CircleModelV2(nn.Module):

    def __init__(self):
        super().__init__()
        # Three linear layers
        self.layer1 = nn.Linear(in_features=2, out_features=15)
        self.layer2 = nn.Linear(in_features=15, out_features=15)
        self.layer3 = nn.Linear(in_features=15, out_features=1)
        
        # Activation functions - these introduce non-linearity!
        # ReLU (Rectified Linear Unit): f(x) = max(0, x)
        # - Outputs x if x > 0, otherwise outputs 0
        # - Simple but very effective
        self.relu = nn.ReLU()
        
        # Tanh: another activation function (not used in this model)
        # f(x) = (e^x - e^-x) / (e^x + e^-x)
        # Outputs values in range (-1, 1)
        self.tanh_fn = nn.Tanh()

    def forward(self, x):
        """
        Forward pass with ReLU activations between layers.
        Flow: x -> linear1 -> ReLU -> linear2 -> ReLU -> linear3 -> output
        
        The ReLU activations allow the model to learn non-linear patterns!
        """
        return self.layer3(self.relu(self.layer2(self.relu(self.layer1(x)))))
    
# Create instance and move to device
model_3 = CircleModelV2().to(device)

In [ ]:
# Display model architecture
print(model_3)

In [ ]:
# Setup loss and optimizer for model with non-linearity
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model_3.parameters(), lr=0.1)

In [ ]:
# Train the model with ReLU activations
# This should perform much better on the circles dataset!
torch.manual_seed(42)
epochs = 1000

# Put all data on target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

for epoch in range(epochs):
    ### TRAINING ###
    model_3.train()
    
    # 1. Forward pass
    y_logits = model_3(X_train).squeeze()
    y_pred = torch.round(torch.sigmoid(y_logits))  # logits -> probabilities -> labels
    
    # 2. Calculate loss and accuracy
    loss = loss_fn(y_logits, y_train)  # BCEWithLogitsLoss uses logits directly
    acc = accuracy_fn(y_true=y_train, y_pred=y_pred)
    
    # 3. Optimizer zero grad
    optimizer.zero_grad()

    # 4. Loss backward
    loss.backward()

    # 5. Optimizer step
    optimizer.step()

    ### TESTING ###
    model_3.eval()
    with torch.inference_mode():
        # 1. Forward pass
        test_logits = model_3(X_test).squeeze()
        test_pred = torch.round(torch.sigmoid(test_logits))
        
        # 2. Calculate loss and accuracy
        test_loss = loss_fn(test_logits, y_test)
        test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

    # Print progress every 100 epochs
    if epoch % 100 == 0:
        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test Loss: {test_loss:.5f}, Test Accuracy: {test_acc:.2f}%")

In [ ]:
# Make predictions with the trained non-linear model
model_3.eval()
with torch.inference_mode():
    y_preds = torch.round(torch.sigmoid(model_3(X_test))).squeeze()

# Compare predictions to true labels
y_preds[:10], y[:10]

In [ ]:
# Compare decision boundaries:
# - model_1: Linear model (no activation functions) - straight line boundary
# - model_3: Non-linear model (with ReLU) - curved boundary that fits circles!
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title("Train")
plot_decision_boundary(model_1, X_train, y_train)  # Linear model - poor fit
plt.subplot(1, 2, 2)
plt.title("Test")
plot_decision_boundary(model_3, X_test, y_test)  # Non-linear model - good fit!

## Part 5: Multi-class Classification

Moving from 2 classes to 4 classes - demonstrates how to handle multiple output categories.

In [ ]:
# Create a toy model with a multi-class dataset
import torch
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split

In [ ]:
# Parameters for multi-class dataset
NUM_CLASSES = 4  # 4 different classes (blobs)
NUM_FEATURES = 2  # 2D data (x, y coordinates)
RANDOM_SEED = 42

In [ ]:
# Create multi-class dataset using make_blobs
# Creates 4 clusters of points in 2D space
X_blob, y_blob = make_blobs(
    n_samples=1000,
    n_features=NUM_FEATURES,  # 2D coordinates
    centers=NUM_CLASSES,  # 4 cluster centers
    cluster_std=1.6,  # Standard deviation of clusters
    random_state=RANDOM_SEED
)

# Convert to tensors
X_blob = torch.from_numpy(X_blob).type(torch.float)
# Note: LongTensor for class labels (required for CrossEntropyLoss)
y_blob = torch.from_numpy(y_blob).type(torch.LongTensor)

# Split data into train and test sets
X_blob_train, X_blob_test, y_blob_train, y_blob_test = train_test_split(
    X_blob, y_blob,
    test_size=0.2,
    random_state=RANDOM_SEED
)

In [ ]:
# Visualize the multi-class dataset
# 4 distinct clusters with different colors
plt.figure(figsize=(7, 7))
plt.scatter(X_blob[:, 0], X_blob[:, 1], c=y_blob, cmap=plt.cm.RdYlBu)
plt.show()

In [ ]:
# Create device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
# Check unique class labels
# Should be [0, 1, 2, 3] for 4 classes
torch.unique(y_blob)

In [ ]:
# Create a multi-class classification model
# Key difference: output_features = 4 (one for each class)
class BlobModelv1(nn.Module):

    def __init__(self, input_featues, output_features, hidden_units=8):
        """
        Multi-class classification model.
        
        Args:
            input_features: Number of input features (2 for 2D coordinates)
            output_features: Number of classes (4 for our blobs)
            hidden_units: Size of hidden layers
        """
        super().__init__()
        # Build a sequential network with ReLU activations
        self.linear_layer_stack = nn.Sequential(
            nn.Linear(in_features=input_featues, out_features=hidden_units),
            nn.ReLU(),  # Non-linearity
            nn.Linear(in_features=hidden_units, out_features=hidden_units),
            nn.ReLU(),  # Non-linearity
            # Output layer: 4 values (one per class)
            nn.Linear(in_features=hidden_units, out_features=output_features),
        )

    def forward(self, x):
        """
        Forward pass returns raw logits for each class.
        For 4 classes, outputs shape will be (batch_size, 4)
        """
        return self.linear_layer_stack(x)
    
# Create model instance
model_4 = BlobModelv1(
    input_featues=2,  # 2D input
    output_features=4,  # 4 classes
    hidden_units=8
).to(device)

In [ ]:
# Setup loss function and optimizer for multi-class classification
# CrossEntropyLoss is used for multi-class classification
# It combines LogSoftmax + NLLLoss in one operation
# Note: Different from BCEWithLogitsLoss (binary classification)
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    params=model_4.parameters(),
    lr=0.1
)

In [ ]:
# Move data to device and test the model
X_blob_train, y_blob_train = X_blob_train.to(device), y_blob_train.to(device)
X_blob_test, y_blob_test = X_blob_test.to(device), y_blob_test.to(device)

model_4.eval()

with torch.inference_mode():
    # Get raw logits from model
    # Shape: (batch_size, 4) - one value per class
    y_logits = model_4(X_blob_train)

In [ ]:
# Convert model's logit outputs to prediction probabilities
# Softmax converts logits to probabilities that sum to 1
# dim=1 means apply softmax across classes (for each sample)
y_preds_probs = torch.softmax(y_logits, dim=1)

print(y_logits[:5])  # Raw logits
print(y_preds_probs[:5])  # Probabilities (each row sums to 1.0)

In [ ]:
# Train the multi-class model
torch.manual_seed(42)

# Set number of epochs
epochs = 100

# Put data to target device
X_blob_train, y_blob_train = X_blob_train.to(device), y_blob_train.to(device)
X_blob_test, y_blob_test = X_blob_test.to(device), y_blob_test.to(device)

for epoch in range(epochs):
    ### TRAINING ###
    model_4.train()

    # 1. Forward pass
    y_logits = model_4(X_blob_train)  # Raw logits for each class
    # Convert to prediction labels:
    # logits -> probabilities (softmax) -> class with highest probability (argmax)
    y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)
    
    # 2. Calculate loss and accuracy
    # CrossEntropyLoss expects raw logits (not probabilities)
    loss = loss_fn(y_logits, y_blob_train)
    acc = accuracy_fn(y_true=y_blob_train, y_pred=y_pred)

    # 3. Optimizer zero grad
    optimizer.zero_grad()

    # 4. Loss backwards
    loss.backward()

    # 5. Optimizer step
    optimizer.step()

    ### TESTING ###
    model_4.eval()
    with torch.inference_mode():
        # 1. Forward pass
        test_logits = model_4(X_blob_test)
        test_pred = torch.softmax(test_logits, dim=1).argmax(dim=1)
        
        # 2. Calculate test loss and accuracy
        test_loss = loss_fn(test_logits, y_blob_test)
        test_acc = accuracy_fn(y_true=y_blob_test, y_pred=test_pred)

    # Print progress every 10 epochs
    if epoch % 10 == 0:
        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Acc: {acc:.2f}% | Test Loss: {test_loss:.5f}, Test Acc: {test_acc:.2f}%")

In [ ]:
# Make predictions with trained model
model_4.eval()
with torch.inference_mode():
    y_logits = model_4(X_blob_test)

# View the first 10 raw logit predictions
# Each row has 4 values (one per class)
y_logits[:10]

In [ ]:
# Convert logits to probabilities and then to class predictions

# Step 1: Logits -> Probabilities using softmax
y_pred_probs = torch.softmax(y_logits, dim=1)

# Step 2: Probabilities -> Class labels using argmax
# argmax returns the index of the highest value (the predicted class)
y_preds = y_pred_probs.argmax(dim=1)

# Compare first 10 predictions to true labels
print(f"Predictions: {y_preds[:10]}\nLabels: {y_blob_test[:10]}")
print(f"Test accuracy: {accuracy_fn(y_true=y_blob_test, y_pred=y_preds)}%")

In [ ]:
# Visualize decision boundaries for multi-class classification
# Should show 4 distinct regions (one for each class)
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title("Train")
plot_decision_boundary(model_4, X_blob_train, y_blob_train)
plt.subplot(1, 2, 2)
plt.title("Test")
plot_decision_boundary(model_4, X_blob_test, y_blob_test)

## Using TorchMetrics for Evaluation

In [ ]:
# TorchMetrics provides pre-built evaluation metrics
# More reliable than custom accuracy functions
from torchmetrics import Accuracy

# Setup metric and ensure it's on the target device
# task='multiclass' specifies multi-class classification
# num_classes=4 tells it we have 4 classes
torchmetrics_accuracy = Accuracy(task='multiclass', num_classes=4).to(device)

# Calculate accuracy using TorchMetrics
# Should match our custom accuracy_fn results
torchmetrics_accuracy(y_preds, y_blob_test)